In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U transformers datasets peft accelerate bitsandbytes bert-score pandas

import os, json, math, gc
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from bert_score import score as bertscore

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.set_grad_enabled(False)


In [ ]:
# ---------- BASE MODEL ----------
MODEL_BASE = "issai/LLama-3.1-KazLLM-1.0-8B"

# ---------- YOUR DRIVE BASE ----------
BASE_PATH = "/content/drive/MyDrive/685_final_project"
if not os.path.exists(BASE_PATH):
    BASE_PATH = "/content/drive/MyDrive/685 final project"

# ---------- DATA ----------
CTRL_TEST_PATH    = os.path.join(BASE_PATH, "data/instruction_test.jsonl")
REALIZER_TEST_PATH = os.path.join(BASE_PATH, "data/realizer_test.jsonl")


REALIZER_SPLIT_TOKEN = "[INST] Write the fairy tale based on this plan.\n"

# ---------- ADAPTERS ----------
INSTR_ADAPTER_DIR   = os.path.join(BASE_PATH, "models/instruction_tuned_kazllm")
REALIZER_ADAPTER_DIR = os.path.join(BASE_PATH, "models/realizer_kazllm")
DPO_ADAPTER_DIR      = os.path.join(BASE_PATH, "models/realizer_dpo_kazllm_v3")
PLANNER_ADAPTER_DIR  = os.path.join(BASE_PATH, "models/planner_kazllm")

# ---------- EVAL LIMITS ----------
N_CTRL   = 100
N_REAL   = 100
N_E2E    = 30

MAX_SEQ_LEN = 2048
GEN_MAX_NEW_TOKENS = 350

print("BASE_PATH:", BASE_PATH)


In [ ]:
import os, glob, re

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def load_prompt_completion(path, split_token=None):
    """Returns list of dicts: {"prompt":..., "completion":...} from either prompt/completion or text."""
    rows = read_jsonl(path)
    out = []
    for r in rows:
        if "prompt" in r and "completion" in r:
            out.append({"prompt": r["prompt"], "completion": r["completion"]})
        elif "text" in r and split_token is not None:
            text = r["text"]
            i = text.find(split_token)
            if i == -1:
                continue
            prompt_end = i + len(split_token)
            prompt = text[:prompt_end]
            comp = text[prompt_end:].lstrip("\n")
            if comp:
                out.append({"prompt": prompt, "completion": comp})
    return out

def build_batch(tokenizer, prompts, completions, max_len=2048):
    """completion-only labels: prompt masked (-100), completion supervised."""
    p_tok = tokenizer(prompts, add_special_tokens=False)
    c_tok = tokenizer(completions, add_special_tokens=False)

    input_ids_list, labels_list, attn_list, n_toks = [], [], [], []

    for p_ids, c_ids in zip(p_tok["input_ids"], c_tok["input_ids"]):
        c_ids = c_ids + [tokenizer.eos_token_id]

        input_ids = p_ids + c_ids
        labels    = ([-100] * len(p_ids)) + c_ids
        attn      = [1] * len(input_ids)

        # truncate: keep completion as much as possible
        if len(input_ids) > max_len:
            overflow = len(input_ids) - max_len
            keep_p = max(len(p_ids) - overflow, 0)
            p_ids = p_ids[-keep_p:]

            input_ids = p_ids + c_ids
            labels    = ([-100] * len(p_ids)) + c_ids
            attn      = [1] * len(input_ids)

            input_ids = input_ids[:max_len]
            labels    = labels[:max_len]
            attn      = attn[:max_len]

        input_ids_list.append(input_ids)
        labels_list.append(labels)
        attn_list.append(attn)
        n_toks.append(sum(1 for x in labels if x != -100))

    # pad to max in batch
    pad_id = tokenizer.pad_token_id
    L = max(len(x) for x in input_ids_list)

    def pad(seq, pad_value):
        return seq + [pad_value] * (L - len(seq))

    input_ids = torch.tensor([pad(x, pad_id) for x in input_ids_list], dtype=torch.long)
    attention = torch.tensor([pad(x, 0) for x in attn_list], dtype=torch.long)
    labels    = torch.tensor([pad(x, -100) for x in labels_list], dtype=torch.long)

    return {"input_ids": input_ids, "attention_mask": attention, "labels": labels, "n_toks": torch.tensor(n_toks)}

@torch.no_grad()
def eval_loss_ppl(model, tokenizer, data, batch_size=1, max_len=2048):
    """Returns avg_loss (per token) and perplexity over completion tokens."""
    model.eval()
    total_nll = 0.0
    total_toks = 0

    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        prompts = [x["prompt"] for x in batch]
        comps   = [x["completion"] for x in batch]

        b = build_batch(tokenizer, prompts, comps, max_len=max_len)
        n_toks = int(b["n_toks"].sum().item())
        if n_toks == 0:
            continue

        b = {k: v.to(model.device) for k, v in b.items() if k in ["input_ids","attention_mask","labels"]}
        out = model(**b)
        loss = float(out.loss.item())  # mean over non -100 tokens

        total_nll += loss * n_toks
        total_toks += n_toks

    if total_toks == 0:
        return float("nan"), float("nan")

    avg_loss = total_nll / total_toks
    ppl = math.exp(avg_loss) if avg_loss < 50 else float("inf")
    return avg_loss, ppl

@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_new_tokens=350, temperature=0.8, top_p=0.9):
    model.eval()
    x = tokenizer(prompt, return_tensors="pt").to(model.device)
    y = model.generate(
        **x,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )[0]
    full = tokenizer.decode(y, skip_special_tokens=True)
    return full[len(prompt):].strip() if full.startswith(prompt) else full.strip()

def eval_bertscore(cands, refs, device="cuda"):
    """Returns mean F1."""
    P, R, F1 = bertscore(
        cands, refs,
        model_type="xlm-roberta-base",
        lang="kk",
        rescale_with_baseline=False,   # baseline для kk часто отсутствует
        verbose=False,
        device=device,
        batch_size=8
    )
    return float(F1.mean().item())

def load_model_with_optional_adapter(adapter_dir=None):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_BASE,
        token=True,
        quantization_config=bnb,
        device_map="auto",
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
        max_memory={0: "21GiB", "cpu": "64GiB"},
    )
    base.config.use_cache = False

    peft_dir = resolve_peft_dir(adapter_dir)
    if peft_dir is None:
        if adapter_dir is None:
            return base
        raise ValueError(f"PEFT adapter not found in: {adapter_dir}")

    return PeftModel.from_pretrained(base, peft_dir)



def find_latest_adapter_dir(root):
  cand = glob.glob(os.path.join(root, "**", "adapter_config.json"), recursive=True)
  if not cand:
      return None

  def ckpt_num(path):
      m = re.search(r"checkpoint-(\d+)", path)
      return int(m.group(1)) if m else 0

  best = max(cand, key=lambda p: ckpt_num(p))
  return os.path.dirname(best)

def resolve_peft_dir(maybe_dir):
    if maybe_dir is None:
        return None
    if os.path.isfile(os.path.join(maybe_dir, "adapter_config.json")):
        return maybe_dir
    # попробуем найти внутри
    cand = glob.glob(os.path.join(maybe_dir, "**", "adapter_config.json"), recursive=True)
    if not cand:
        return None
    def ckpt_num(path):
        m = re.search(r"checkpoint-(\d+)", path)
        return int(m.group(1)) if m else 0
    best = max(cand, key=lambda p: ckpt_num(p))
    return os.path.dirname(best)



In [ ]:
DPO_ROOT = "/content/drive/MyDrive/685_final_project/models/realizer_dpo_kazllm_v3"
DPO_ADAPTER_DIR = find_latest_adapter_dir(DPO_ROOT)

print("Resolved DPO adapter dir:", DPO_ADAPTER_DIR)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE, token=True, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

ctrl_test = load_prompt_completion(CTRL_TEST_PATH, split_token=None)
real_test = load_prompt_completion(REALIZER_TEST_PATH, split_token=REALIZER_SPLIT_TOKEN)

ctrl_test = ctrl_test[:N_CTRL]
real_test = real_test[:N_REAL]

print("CTRL test examples:", len(ctrl_test))
print("REALIZER test examples:", len(real_test))
print("\nCTRL sample keys:", ctrl_test[0].keys())
print("REAL sample keys:", real_test[0].keys())


In [ ]:
results = []

# --- BASE ---
base_model = load_model_with_optional_adapter(None)
base_model.config.pad_token_id = tokenizer.eos_token_id

loss, ppl = eval_loss_ppl(base_model, tokenizer, ctrl_test, batch_size=1, max_len=MAX_SEQ_LEN)

# BERTScore (generation vs reference completion)
cands = [generate_text(base_model, tokenizer, x["prompt"], max_new_tokens=GEN_MAX_NEW_TOKENS) for x in ctrl_test[:min(30, len(ctrl_test))]]
refs  = [x["completion"] for x in ctrl_test[:min(30, len(ctrl_test))]]
bs = eval_bertscore(cands, refs, device="cuda" if torch.cuda.is_available() else "cpu")

results.append({"task":"CTRL-only", "model":"BASE", "loss":loss, "ppl":ppl, "bertscore_f1":bs})

# --- INSTRUCTION-TUNED (adapter) ---
instr_model = load_model_with_optional_adapter(INSTR_ADAPTER_DIR)
instr_model.config.pad_token_id = tokenizer.eos_token_id

loss, ppl = eval_loss_ppl(instr_model, tokenizer, ctrl_test, batch_size=1, max_len=MAX_SEQ_LEN)
cands = [generate_text(instr_model, tokenizer, x["prompt"], max_new_tokens=GEN_MAX_NEW_TOKENS) for x in ctrl_test[:min(30, len(ctrl_test))]]
refs  = [x["completion"] for x in ctrl_test[:min(30, len(ctrl_test))]]
bs = eval_bertscore(cands, refs, device="cuda" if torch.cuda.is_available() else "cpu")

results.append({"task":"CTRL-only", "model":"INSTRUCTION_TUNED", "loss":loss, "ppl":ppl, "bertscore_f1":bs})

pd.DataFrame(results)




In [ ]:
# Clear big models
del base_model, instr_model
gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()

# --- BASE on REALIZER prompts (CTRL+PLAN) ---
base_model = load_model_with_optional_adapter(None)
base_model.config.pad_token_id = tokenizer.eos_token_id

loss, ppl = eval_loss_ppl(base_model, tokenizer, real_test, batch_size=1, max_len=MAX_SEQ_LEN)
cands = [generate_text(base_model, tokenizer, x["prompt"], max_new_tokens=GEN_MAX_NEW_TOKENS) for x in real_test[:min(30, len(real_test))]]
refs  = [x["completion"] for x in real_test[:min(30, len(real_test))]]
bs = eval_bertscore(cands, refs, device="cuda" if torch.cuda.is_available() else "cpu")
results.append({"task":"REALIZER(CTRL+PLAN)", "model":"BASE", "loss":loss, "ppl":ppl, "bertscore_f1":bs})

# --- REALIZER SFT adapter ---
real_sft = load_model_with_optional_adapter(REALIZER_ADAPTER_DIR)
real_sft.config.pad_token_id = tokenizer.eos_token_id

loss, ppl = eval_loss_ppl(real_sft, tokenizer, real_test, batch_size=1, max_len=MAX_SEQ_LEN)
cands = [generate_text(real_sft, tokenizer, x["prompt"], max_new_tokens=GEN_MAX_NEW_TOKENS) for x in real_test[:min(30, len(real_test))]]
refs  = [x["completion"] for x in real_test[:min(30, len(real_test))]]
bs = eval_bertscore(cands, refs, device="cuda" if torch.cuda.is_available() else "cpu")
results.append({"task":"REALIZER(CTRL+PLAN)", "model":"REALIZER_SFT", "loss":loss, "ppl":ppl, "bertscore_f1":bs})

In [ ]:
# Clear big models
for v in ["base_model", "instr_model", "real_sft", "dpo_model", "planner", "realizer"]:
    if v in globals():
        del globals()[v]

import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()



# --- DPO adapter (поверх realizer) ---
DPO_ADAPTER_DIR = find_latest_adapter_dir("/content/drive/MyDrive/685_final_project/models/realizer_dpo_kazllm_v3")
print("Using DPO adapter:", DPO_ADAPTER_DIR)
dpo_model = load_model_with_optional_adapter(DPO_ADAPTER_DIR)
dpo_model.config.pad_token_id = tokenizer.eos_token_id

loss, ppl = eval_loss_ppl(dpo_model, tokenizer, real_test, batch_size=1, max_len=MAX_SEQ_LEN)
cands = [generate_text(dpo_model, tokenizer, x["prompt"], max_new_tokens=GEN_MAX_NEW_TOKENS) for x in real_test[:min(30, len(real_test))]]
refs  = [x["completion"] for x in real_test[:min(30, len(real_test))]]
bs = eval_bertscore(cands, refs, device="cuda" if torch.cuda.is_available() else "cpu")
results.append({"task":"REALIZER(CTRL+PLAN)", "model":"DPO", "loss":loss, "ppl":ppl, "bertscore_f1":bs})

pd.DataFrame(results)


In [ ]:
# Load planner + realizer models (adapters)
# Clear big models
for v in ["base_model", "instr_model", "real_sft", "dpo_model", "planner", "realizer"]:
    if v in globals():
        del globals()[v]

import gc, torch
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

planner = load_model_with_optional_adapter(PLANNER_ADAPTER_DIR)
planner.config.pad_token_id = tokenizer.eos_token_id

realizer = load_model_with_optional_adapter(REALIZER_ADAPTER_DIR)
realizer.config.pad_token_id = tokenizer.eos_token_id

def extract_ctrl_from_realizer_prompt(p):
    # prompt starts with [CTRL] ... [PLAN] ... [INST] ...
    # planner expects: "[CTRL] tags [INST] Generate a structured plan.\n"
    if "[CTRL]" not in p:
        return None
    # take substring between [CTRL] and [PLAN]
    try:
        s = p.split("[CTRL]",1)[1]
        tags = s.split("[PLAN]",1)[0].strip()
        return tags
    except Exception:
        return None

@torch.no_grad()
def generate_plan(tags):
    planner_prompt = f"[CTRL] {tags} [INST] Generate a structured plan.\n"
    plan = generate_text(planner, tokenizer, planner_prompt, max_new_tokens=400, temperature=0.7, top_p=0.9)
    # try keep only plan block if exists
    if "[PLAN_START]" in plan and "[PLAN_END]" in plan:
        plan = plan.split("[PLAN_START]",1)[1]
        plan = "[PLAN_START]" + plan.split("[PLAN_END]",1)[0] + "[PLAN_END]"
    return plan.strip()

def build_realizer_prompt(tags, plan):
    return f"[CTRL] {tags} [PLAN] {plan} [INST] Write the fairy tale based on this plan.\n"

e2e_samples = real_test[:min(N_E2E, len(real_test))]

e2e_prompts = []
e2e_refs = []
e2e_cands = []

for ex in e2e_samples:
    tags = extract_ctrl_from_realizer_prompt(ex["prompt"])
    if tags is None:
        continue
    plan = generate_plan(tags)
    rp = build_realizer_prompt(tags, plan)
    story = generate_text(realizer, tokenizer, rp, max_new_tokens=GEN_MAX_NEW_TOKENS)
    e2e_prompts.append(rp)
    e2e_refs.append(ex["completion"])
    e2e_cands.append(story)

print("E2E evaluated:", len(e2e_cands))

# BERTScore
bs = eval_bertscore(e2e_cands, e2e_refs, device="cuda" if torch.cuda.is_available() else "cpu")

# Pipeline loss/ppl: NLL(reference story | generated plan prompt)
pipeline_data = [{"prompt": p, "completion": r} for p, r in zip(e2e_prompts, e2e_refs)]
loss, ppl = eval_loss_ppl(realizer, tokenizer, pipeline_data, batch_size=1, max_len=MAX_SEQ_LEN)

results.append({"task":"PIPELINE(planner->realizer)", "model":"PLANNER+REALIZER(SFT)", "loss":loss, "ppl":ppl, "bertscore_f1":bs})

pd.DataFrame(results)


In [ ]:
pd.DataFrame(results).to_csv('/content/drive/MyDrive/685_final_project/data/evaluation_part1.csv')